In [31]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt


In [33]:
def create_windows(data, size):
    return np.array([data[i:i+size] for i in range(len(data)-size)])
    """
    questo serve perchè i campionamenti funzionano come una coda con dimensione limitata al numero di campionamenti,
    quindi per simulare lo scorrimento si selezionano anche elementi ripetuti.
    """

dimensione_finestra = 10
df = pd.read_csv("capture_barca.csv").iloc[10505:, 1:3]
scaler = StandardScaler()
data_scaled = scaler.fit_transform(df)

train_data, test_data = train_test_split(data_scaled, test_size=0.2, shuffle=False)

X_train = create_windows(train_data, dimensione_finestra)
X_test = create_windows(test_data, dimensione_finestra)

print(X_train)

[[[-0.62151918 -0.37196759]
  [-0.35413169 -0.19132831]
  [ 0.10680273 -0.29403772]
  ...
  [ 0.03597932  0.15791961]
  [-0.32883005  0.19189272]
  [-0.66365328  0.23671137]]

 [[-0.35413169 -0.19132831]
  [ 0.10680273 -0.29403772]
  [ 0.55445115 -0.15509271]
  ...
  [-0.32883005  0.19189272]
  [-0.66365328  0.23671137]
  [-1.05159407  0.37511769]]

 [[ 0.10680273 -0.29403772]
  [ 0.55445115 -0.15509271]
  [ 0.03793782 -0.30430866]
  ...
  [-0.66365328  0.23671137]
  [-1.05159407  0.37511769]
  [-0.71430949  0.60319723]]

 ...

 [[ 0.22735604  0.42370715]
  [ 0.25408684  0.10638534]
  [ 0.0420136  -0.16705154]
  ...
  [-0.14068221  0.19972162]
  [ 0.18392509  0.0830782 ]
  [ 0.29862937  0.19541213]]

 [[ 0.25408684  0.10638534]
  [ 0.0420136  -0.16705154]
  [-0.13970296 -0.38399824]
  ...
  [ 0.18392509  0.0830782 ]
  [ 0.29862937  0.19541213]
  [ 0.48862984  0.14129217]]

 [[ 0.0420136  -0.16705154]
  [-0.13970296 -0.38399824]
  [-1.54500246 -1.79176361]
  ...
  [ 0.29862937  0.195412

In [34]:
input_shape = (dimensione_finestra, 2)
autoencoder = models.Sequential([
  layers.Input(shape=input_shape),
  layers.GaussianNoise(0.05),
  layers.Conv1D(16, 3, padding='same'),
  layers.BatchNormalization(),
  layers.Activation('relu6'),
  layers.MaxPooling1D(2),
  layers.Conv1D(8, 3, activation='relu6', padding='same'),
  layers.UpSampling1D(2),
  layers.Conv1D(16, 3, activation='relu6', padding='same'),
  layers.Conv1D(2, 3, activation='linear', padding='same')
 ])

lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    verbose=1
)


autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.fit(X_train, X_train, epochs=150, batch_size=32, callbacks=lr_scheduler , validation_split=0.1, verbose=1)
autoencoder.save("autoencoder_barca_v1.keras")


Epoch 1/150
144/144 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - loss: 0.4211 - val_loss: 0.3013 - learning_rate: 0.0010
Epoch 2/150
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2033 - val_loss: 0.1879 - learning_rate: 0.0010
Epoch 3/150
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1624 - val_loss: 0.1414 - learning_rate: 0.0010
Epoch 4/150
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1350 - val_loss: 0.1112 - learning_rate: 0.0010
Epoch 5/150
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1140 - val_loss: 0.0958 - learning_rate: 0.0010
Epoch 6/150
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1007 - val_loss: 0.0840 - learning_rate: 0.0010
Epoch 7/150
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0922 - val_loss: 0.0772 - learning_rate: 0.0010
Epoch 8/150
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0860 - val_loss: 0.0677 - learning_rate: 0.0010
Epoch 9/150
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0776 - val_loss: 0.0632 - learning_rate: 0.0010


In [40]:
#X_anomalie[1]
esempio = np.array([[ 0.0965868, -1.92708147], [ 0.34528919, -1.1406003 ],
                    [-1.01694989, -2.31529432], [-0.6528551,  -1.23185367],
                    [ 0.26120624, -0.60385384], [ 0.71414789, -0.06549132],
                    [ 0.91431136,  0.48885221], [ 1.51906283,  1.90164532],
                    [-0.87104202,  2.35098104], [-0.59071258, -0.91981098]])

#X_train[1]
esempio2 = np.array([[-0.62151918, -0.37196759],
                    [-0.35413169, -0.19132831],
                    [ 0.10680273, -0.29403772],
                    [ 0.55445115, -0.15509271],
                    [ 0.03793782, -0.30430866],
                    [-0.74217834, -0.70548588],
                    [ 1.13877075, -1.45095511],
                    [ 0.03597932,  0.15791961],
                    [-0.32883005,  0.19189272],
                    [-0.66365328,  0.23671137]])

input_batch = np.expand_dims(esempio2, axis=0) # Trasforma (10, 2) in (1, 10, 2)
ricostruzione = autoencoder(input_batch, training=False).numpy()
mse_quiete = np.mean(np.square(X_train - autoencoder.predict(X_train)), axis=(1, 2))
threshold = np.percentile(mse_quiete, 98)

mse_singolo = np.mean(np.square(input_batch - ricostruzione))
if(mse_singolo > threshold):
    print("Anomalia rilevata")
else:
    print("Stato Normale")
print(f"\nMSE Singola Finestra: {mse_singolo:.6f}")

print(X_train[0])

160/160 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Stato Normale

MSE Singola Finestra: 0.003739
[[-0.62151918 -0.37196759]
 [-0.35413169 -0.19132831]
 [ 0.10680273 -0.29403772]
 [ 0.55445115 -0.15509271]
 [ 0.03793782 -0.30430866]
 [-0.74217834 -0.70548588]
 [ 1.13877075 -1.45095511]
 [ 0.03597932  0.15791961]
 [-0.32883005  0.19189272]
 [-0.66365328  0.23671137]]


In [ ]:
import tensorflow as tf
import numpy as np


def convert_to_tflite_int8(model, X_calib, output_path="autoencoder.tflite"):

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    calib_samples = X_calib[np.random.choice(len(X_calib),
                             size=min(500, len(X_calib)),
                             replace=False)]

    def representative_dataset():
        for sample in calib_samples:
            yield [sample[np.newaxis, ...].astype(np.float32)]

    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type  = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()

    with open(output_path, 'wb') as f:
        f.write(tflite_model)

    print(f"Modello salvato: {output_path}")
    print(f"Dimensione: {len(tflite_model) / 1024:.1f} KB")
    return tflite_model


def verify_quantized_model(tflite_model, X_test, threshold_float):
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()

    inp  = interpreter.get_input_details()[0]
    out  = interpreter.get_output_details()[0]

    in_scale, in_zp   = inp['quantization']
    out_scale, out_zp = out['quantization']

    mse_list = []
    for window in X_test:
        x_int8 = np.clip(
            np.round(window / in_scale + in_zp), -128, 127
        ).astype(np.int8)[np.newaxis, ...]

        interpreter.set_tensor(inp['index'], x_int8)
        interpreter.invoke()

        y_int8  = interpreter.get_tensor(out['index'])[0]
        y_float = (y_int8.astype(np.float32) - out_zp) * out_scale

        mse = np.mean((window - y_float) ** 2)
        mse_list.append(mse)

    mse_arr   = np.array(mse_list)
    anomalies = mse_arr > threshold_float
    print(f"Anomalie modello quantizzato: {np.sum(anomalies)} su {len(mse_arr)}")

    return mse_arr

def measure_arena_size(tflite_model):
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()

    total_bytes = 0
    print(f"\n{'Tensor':<50} {'Shape':<20} {'Dtype':<10} {'Bytes'}")
    print("-" * 95)

    for t in interpreter.get_tensor_details():
        shape   = t['shape']
        dtype   = t['dtype']
        n_bytes = int(np.prod(shape)) * np.dtype(dtype).itemsize
        total_bytes += n_bytes
        print(f"{t['name']:<50} {str(shape):<20} {str(dtype):<10} {n_bytes}")

    print(f"\nTotale tensori: {total_bytes} bytes ({total_bytes/1024:.1f} KB)")
    print(f"\nArena size consigliata (con overhead 20%): "
          f"{int(total_bytes * 1.2)} bytes ({total_bytes * 1.2 / 1024:.1f} KB)")

    return total_bytes

measure_arena_size(tflite_model)


tflite_model = convert_to_tflite_int8(autoencoder, X_train)
arena_bytes  = measure_arena_size(tflite_model)
mse_quant    = verify_quantized_model(tflite_model, X_test, threshold)

print(f"\nScaler mean:  {scaler.mean_}")
print(f"Scaler scale: {scaler.scale_}")